# Réseaux de neurones récurrents (RNN) : génération de texte et analyse de sentiment

**Module :** Machine Learning 2 — ECE Paris, 2025/2026  
**Auteur :** Édouard Menut

Deux applications des couches récurrentes (GRU) avec TensorFlow / Keras :

1. **Génération de texte caractère par caractère** : un modèle entraîné sur les œuvres de Shakespeare apprend à prédire le caractère suivant, puis écrit un texte à partir de « ROMEO: ».
2. **Classification de sentiment** : un réseau Embedding + GRU classe des critiques de films IMDB en positives ou négatives.

> Données non incluses dans le dépôt (fichiers volumineux fournis en cours) : `shakespeare.txt` (corpus *Tiny Shakespeare* du tutoriel TensorFlow, ~1,1 M caractères) et `MovieReview.csv` (50 000 critiques IMDB étiquetées).

---
## Partie 1 — Génération de texte

### Préparation du corpus
On lit le texte, on construit le vocabulaire des caractères uniques et on convertit chaque caractère en indice entier.

In [4]:
with open("shakespeare.txt", encoding="utf-8") as file:
    text = file.read()

print(text[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [5]:
vocab = sorted(set(text))

print(len(vocab))

65


In [7]:
import numpy as np
print('{')
for char, _ in zip(char2idx, range(10)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

{
  '\n':   0,
  ' ' :   1,
  '!' :   2,
  '$' :   3,
  '&' :   4,
  "'" :   5,
  ',' :   6,
  '-' :   7,
  '.' :   8,
  '3' :   9,
  ...
}


In [8]:
text_as_int = [char2idx[char] for char in text]
text_as_int[:10]

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47]

### Construction du dataset TensorFlow
Le texte est découpé en séquences de 101 caractères. Chaque séquence donne une entrée (100 premiers caractères) et une cible décalée d'un caractère : le modèle apprend à prédire, à chaque position, le caractère suivant.

Exemple : `"How are you"` → entrée `"How are yo"`, cible `"ow are you"`.

In [2]:
import tensorflow as tf
seq_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
char_dataset

In [10]:
idx2char = np.array(vocab)

for i in char_dataset.take(5):
     print(idx2char[i.numpy()])

F
i
r
s
t


In [11]:
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

In [12]:
for item in sequences.take(2):
    print(repr(''.join(idx2char[item.numpy()])))

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
'are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you k'


In [13]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

In [14]:
for input_example, target_example in  dataset.take(1):
    print ('Input data: ', repr(''.join(idx2char[input_example.numpy()])))
    print ('Target data:', repr(''.join(idx2char[target_example.numpy()])))

Input data:  'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
Target data: 'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


In [15]:
dataset = dataset.shuffle(10000).batch(64, drop_remainder=True)

### Modèle
- **Embedding** (256 dimensions) : représente chaque caractère par un vecteur appris.
- **GRU** de 512 unités, `return_sequences=True` pour prédire à chaque pas de temps, `stateful=True` pour conserver l'état caché d'un batch à l'autre.
- **Dense softmax** sur les 65 caractères du vocabulaire.

Entraînement : Adam (lr = 1e-3), entropie croisée, 30 epochs.

In [16]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, RNN, GRUCell

vocab_size = len(vocab)

def build_model(batch_size):

    model = tf.keras.Sequential()

    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=256
    ))

    model.add(RNN(
        GRUCell(512),
        return_sequences=True,
        stateful=True
    ))

    model.add(Dense(
        vocab_size,
        activation='softmax'
    ))

    return model

In [17]:
model = build_model(batch_size=64)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy'
)

In [18]:
history = model.fit(
    dataset,
    epochs=30
)

Epoch 1/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - loss: 2.9595
Epoch 2/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 1.9817
Epoch 3/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 1.7353
Epoch 4/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 1.5939
Epoch 5/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - loss: 1.5105
Epoch 6/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - loss: 1.4568
Epoch 7/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.4164
Epoch 8/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.3861
Epoch 9/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.3567
Epoch 10/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 12s 58ms/step - loss: 1.3390
Epoch 11/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - loss: 1.3220
Epoch 12/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.3060
Epoch 13/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.2888
Epoch 14/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.2731
Epoch 15/30
172/172 ━━━━━━━━━━━━━━━

### Génération
Le modèle a été entraîné avec des batchs de 64 et un état stateful. Pour générer un texte seul, on recopie les poids dans un modèle identique de `batch_size = 1`, puis on prédit les caractères un par un en réinjectant chaque prédiction en entrée.

In [21]:
import numpy as np

model.save_weights("temp_weights.weights.h5")

model_single = build_model(batch_size=1)

dummy_input = np.zeros((1, 1), dtype=np.int32)
model_single(dummy_input)

model_single.load_weights("temp_weights.weights.h5")

model_single.save_weights("model_rnn.weights.h5")

In [23]:
def generate_text(model, start_string, num_generate=500):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []
    for layer in model.layers:
        if hasattr(layer, "reset_states"):
            layer.reset_states()

    for i in range(num_generate):
        prediction = model(input_eval)
        prediction = prediction[:, -1, :]
        index = tf.argmax(prediction, axis=-1).numpy()[0]
        input_eval = tf.expand_dims([index], 0)
        text_generated.append(idx2char[index])
    return start_string + ''.join(text_generated)

print(generate_text(model, start_string="ROMEO:"))

ROMEO:
I think there be the man of the fardel of my son,
And therefore I'll swear to thee, the deed that thou hast made thee to the good of the world.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done, the time so much to death.

DUKE VINCENTIO:
I will not be done


La loss passe de 2,96 à la première epoch à 1,09 après 30 epochs. Le texte généré respecte la structure d'une pièce de théâtre (noms de personnages en majuscules suivis de `:`, retours à la ligne, ponctuation) et produit des mots anglais corrects.

**Limite observée :** le texte finit par boucler (« DUKE VINCENTIO: I will not be done… » répété). La génération choisit toujours le caractère le plus probable (`argmax`, décodage glouton). Un **échantillonnage avec température** (`tf.random.categorical`) donnerait un texte plus varié.

---
## Partie 2 — Analyse de sentiment sur des critiques IMDB

### Chargement et nettoyage
Nettoyage de chaque critique : passage en minuscules, suppression des accents, de la ponctuation, des mots de moins de 3 lettres et des *stopwords* anglais (NLTK).

In [25]:
import pandas as pd
df = pd.read_csv("MovieReview.csv")

print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [27]:
import re
import unicodedata
import pandas as pd

try:
    import swifter
    use_swifter = True
except ModuleNotFoundError:
    use_swifter = False
    print("Swifter not installed. Using pandas apply instead.")

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, PunktSentenceTokenizer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

stop_words = set(stopwords.words('english'))

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'\s+', ' ', w)
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    w = re.sub(r'\b\w{0,2}\b', '', w)
    try:
        mots = word_tokenize(w.strip())
    except LookupError:
        tokenizer = PunktSentenceTokenizer()
        mots = []
        for sentence in tokenizer.tokenize(w.strip()):
            mots.extend(sentence.split())
    mots = [mot for mot in mots if mot not in stop_words]

    return ' '.join(mots).strip()

if use_swifter:
    df.review = df.review.swifter.apply(preprocess_sentence)
else:
    df.review = df.review.apply(preprocess_sentence)

df.head()

,review,sentiment
0,one reviewers mentioned watching episode hooke...,positive
1,wonderful little production . filming techniqu...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake thinks zombie...,negative
4,petter mattei love time money visually stunnin...,positive


### Découpage, tokenisation et padding
- Découpage train / test (10 000 critiques pour l'entraînement, 40 000 pour le test).
- `Tokenizer` limité aux 10 000 mots les plus fréquents, avec un jeton `<OOV>` pour les mots inconnus.
- Chaque critique devient une séquence d'indices, complétée ou tronquée à 500 mots (`pad_sequences`).

In [28]:
from sklearn.model_selection import train_test_split

X_text = df['review']
y = df['sentiment']

X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.8,
    random_state=1234
)

print("Taille X_train :", len(X_text_train))
print("Taille X_test :", len(X_text_test))

Taille X_train : 10000
Taille X_test : 40000


In [29]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=10000,
    lower=True,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_text_train)

print("Nombre de mots dans le dictionnaire :", len(tokenizer.word_index))

Nombre de mots dans le dictionnaire : 50613


In [30]:
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = tokenizer.num_words
print("Exemple word2idx :", list(word2idx.items())[:5])
print("Exemple idx2word :", list(idx2word.items())[:5])
print("Taille du vocabulaire :", vocab_size)

Exemple word2idx : [('<OOV>', 1), ('movie', 2), ('film', 3), ('one', 4), ('like', 5)]
Exemple idx2word : [(1, '<OOV>'), (2, 'movie'), (3, 'film'), (4, 'one'), (5, 'like')]
Taille du vocabulaire : 10000


In [31]:
X_train = tokenizer.texts_to_sequences(X_text_train)
X_test = tokenizer.texts_to_sequences(X_text_test)

In [32]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(
    X_train,
    maxlen=500,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=500,
    padding='post',
    truncating='post'
)

print("Shape de X_train :", X_train.shape)
print("Shape de X_test :", X_test.shape)

Shape de X_train : (10000, 500)
Shape de X_test : (40000, 500)


### Modèle Embedding + GRU
1. **Embedding** (10 000 mots → vecteurs de 200 dimensions)
2. **GRU** de 128 unités qui parcourt la séquence et produit une représentation contextualisée de chaque mot
3. **Dropout** (0,5) contre le sur-apprentissage
4. **GlobalAveragePooling1D** : moyenne sur toute la séquence
5. **Dense** 256 ReLU + Dropout, puis **softmax** à 2 classes

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, RNN, GRUCell, Dropout, GlobalAveragePooling1D, Dense

model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=200))

model.add(RNN(GRUCell(128), return_sequences=True))

model.add(Dropout(0.5))

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation='relu'))

model.add(Dropout(0.5))

model.add(Dense(2, activation='softmax'))

In [34]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model.pop()
model.add(Dense(2, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_enc,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

loss, acc = model.evaluate(X_test, y_test_enc)
print("Test accuracy :", acc)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 81ms/step - accuracy: 0.4909 - loss: 0.6980 - val_accuracy: 0.5205 - val_loss: 0.6902
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.5579 - loss: 0.6729 - val_accuracy: 0.7605 - val_loss: 0.5310
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.8553 - loss: 0.3703 - val_accuracy: 0.8735 - val_loss: 0.3211
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.9409 - loss: 0.1763 - val_accuracy: 0.8670 - val_loss: 0.4010
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 70ms/step - accuracy: 0.9772 - loss: 0.0833 - val_accuracy: 0.8570 - val_loss: 0.4708
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.8555 - loss: 0.4802
Test accuracy : 0.8605250120162964


In [35]:
loss, acc = model.evaluate(X_test, y_test_enc)
print("Test accuracy :", acc)

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.8555 - loss: 0.4802
Test accuracy : 0.8605250120162964


Le modèle atteint **86 % d'accuracy sur 40 000 critiques jamais vues**, en n'ayant été entraîné que sur 10 000 exemples.

### Test sur de nouvelles phrases

In [36]:
new_sentences = [
    "I loved this movie! It was amazing.",
    "The product is terrible and broke immediately.",
    "Not bad, could be better."
]

new_sentences_clean = [preprocess_sentence(s) for s in new_sentences]

new_sequences = tokenizer.texts_to_sequences(new_sentences_clean)

new_padded = pad_sequences(
    new_sequences,
    maxlen=500,
    padding='post',
    truncating='post'
)

predictions = model.predict(new_padded)

predicted_classes = np.argmax(predictions, axis=1)

predicted_labels = le.inverse_transform(predicted_classes)

for sentence, label in zip(new_sentences, predicted_labels):
    print(f"Sentence: {sentence}")
    print(f"Predicted sentiment: {label}")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 715ms/step
Sentence: I loved this movie! It was amazing.
Predicted sentiment: positive
--------------------------------------------------
Sentence: The product is terrible and broke immediately.
Predicted sentiment: negative
--------------------------------------------------
Sentence: Not bad, could be better.
Predicted sentiment: negative
--------------------------------------------------


Les deux premières phrases sont correctement classées. « Not bad, could be better. » est prédite négative : la négation « not » fait partie des stopwords supprimés au nettoyage, et le modèle ne voit plus que « bad » et « better ». Supprimer les stopwords est utile pour un sac de mots, mais **retire des informations essentielles pour un modèle séquentiel** : garder les négations améliorerait ce type de cas.

## Conclusion

| Tâche | Modèle | Résultat |
|---|---|---|
| Génération de texte (Shakespeare) | Embedding + GRU 512 stateful | loss 2,96 → 1,09 ; texte structuré mais répétitif en décodage glouton |
| Sentiment IMDB | Embedding + GRU 128 + pooling | **86 %** d'accuracy sur 40 000 critiques de test |